# 03 — MLflow Registry, Quality Gate Automatizado, Serving e Drift (PSI)
**Hipótese de Governança:** A promoção automatizada via *Quality Gate* contra o baseline heurístico, combinada à monitorização contínua de *drift* (PSI e KS) e auditoria de inferências, assegura confiabilidade estatística em produção e delimita quando o deslocamento de distribuição exige investigação ou retreinamento sob supervisão *Human-in-the-Loop*.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import settings
from src.pipeline.register import promote_approved_model
from src.monitoring.drift_detector import run_drift_audit

mv = promote_approved_model()
reg = {
    "model_name": mv.name,
    "version": mv.version,
    "current_stage": mv.current_stage,
    "run_id": mv.run_id,
}
print("Resultado do Quality Gate e Registro MLflow:")
print(json.dumps(reg, indent=2, ensure_ascii=False, default=str))


C:\tech-challenge-fase5-grupo\.venv\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


C:\tech-challenge-fase5-grupo\.venv\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_api" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Registered model 'passos_magicos_risk_classifier' already exists. Creating a new version of this model...
2026/11/24 07:25:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 38efb3ba73ae49e38c401de10f20b82b has no artifacts at artifact path 'model', registering model based on models:/m-ef48c77098c047b5921376a0905e4489 instead


[Model Registry] Modelo 'passos_magicos_risk_classifier' v2 (Run 38efb3ba73ae49e38c401de10f20b82b) promovido para PRODUCTION!
Resultado do Quality Gate e Registro MLflow:
{
  "model_name": "passos_magicos_risk_classifier",
  "version": 2,
  "current_stage": "None",
  "run_id": "38efb3ba73ae49e38c401de10f20b82b"
}


Created version '2' of model 'passos_magicos_risk_classifier'.
C:\tech-challenge-fase5-grupo\passos-magicos-mlops\src\pipeline\register.py:38: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


## 1. Inspeção Estruturada do Quality Gate e Métricas de Promoção
Leitura tabular dos critérios avaliados no *Quality Gate* e verificação de consistência com os artefatos persistidos em `artifacts/`.

In [2]:
if isinstance(reg, dict):
    display(pd.json_normalize(reg, sep=".").T.rename(columns={0: "valor"}))

qg_path = ROOT / "artifacts" / "quality_gate_report.json"
if qg_path.exists():
    qg_report = json.loads(qg_path.read_text(encoding="utf-8"))
    print("Chaves em quality_gate_report.json:", list(qg_report.keys()))

,valor
model_name,passos_magicos_risk_classifier
version,2
current_stage,None
run_id,38efb3ba73ae49e38c401de10f20b82b


Chaves em quality_gate_report.json: ['champion_model', 'champion_run_id', 'approved', 'checks', 'val_metrics', 'test_oot_metrics', 'baseline_val_metrics']


## 2. Auditoria de Drift Temporal (PSI e Estatística KS)
Avaliação da estabilidade das features entre janelas, destacando o impacto da reestruturação metodológica de 2025 e variáveis com *drift* severo ($	ext{PSI} \ge 0.25$).

In [3]:
ref_df = pd.read_parquet(settings.REFERENCE_DIR / "reference_train.parquet")
cur_df = pd.read_parquet(settings.PROCESSED_DIR / "test_2024_2025.parquet")
drift = run_drift_audit(ref_df, cur_df, "oot_2024_2025")
if isinstance(drift, dict) and "oot_2024_2025" in drift:
    oot = drift["oot_2024_2025"]
    print("Features com drift severo (2024 -> 2025):", oot.get("severe_features", []))
    if isinstance(oot.get("features"), dict):
        df_drift = pd.DataFrame.from_dict(oot["features"], orient="index")
        display(df_drift.sort_values(by=df_drift.columns[0], ascending=False).head(15) if len(df_drift) else df_drift)
else:
    print(json.dumps(drift, indent=2, ensure_ascii=False, default=str))


{
  "window": "oot_2024_2025",
  "timestamp_utc": "2026-11-24T10:25:18.874044+00:00",
  "retrain_triggered": true,
  "severe_drift_count": 4,
  "moderate_drift_count": 8,
  "severe_features": [
    "anos_na_ong",
    "defasagem",
    "ips",
    "ing"
  ],
  "moderate_features": [
    "fase_num",
    "idade",
    "inde",
    "ieg",
    "mat",
    "por",
    "delta_pedra_1y",
    "media_psico"
  ],
  "features": {
    "fase_num": {
      "psi": 0.1921,
      "status": "MODERADO_INVESTIGAR",
      "ks_statistic": 0.1817,
      "ks_pvalue": 0.0,
      "ref_mean": 1.756,
      "cur_mean": 2.579
    },
    "idade": {
      "psi": 0.1212,
      "status": "MODERADO_INVESTIGAR",
      "ks_statistic": 0.1086,
      "ks_pvalue": 0.0019,
      "ref_mean": 11.596,
      "cur_mean": 12.347
    },
    "anos_na_ong": {
      "psi": 5.5508,
      "status": "SEVERO_RETREINAR",
      "ks_statistic": 0.4561,
      "ks_pvalue": 0.0,
      "ref_mean": 1.505,
      "cur_mean": 2.167
    },
    "defasagem": {

## 3. Rigor Operacional: Serving, Limitações e Human-in-the-Loop
* **Contratos de Serving e Auditoria:** A API (`FastAPI` + `Pydantic`) valida intervalos e tipos na entrada e regista eventos em `artifacts/inference_audit_log.jsonl`, permitindo rastreabilidade de predições.
* **Risco de Degradação Pós-Shift:** Quando ocorre *drift* severo em variáveis educacionais ou mudança de regra institucional, o escore probabilístico pode perder calibração mesmo que a arquitetura do modelo permaneça intacta.
* **Decisão Responsável:** O sistema atua como suporte à decisão psicopedagógica (*Human-in-the-Loop*), exigindo revisão humana e reavaliação periódica do modelo e do limiar.